# Presentation figures

Regenerates compact figures used in `presentation/master_thesis_training_set_optimization.tex` from saved thesis outputs.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "outputs").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch, Rectangle
import numpy as np
import pandas as pd
import seaborn as sns

from src.mlp_avggrm_viz_report import prepare_report_data, build_full_baseline_comparison

OUT = ROOT / "figures" / "presentation"
OUT.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "axes.titleweight": "bold",
    "axes.labelsize": 12,
    "xtick.labelsize": 9,
    "ytick.labelsize": 10,
    "legend.fontsize": 9,
    "figure.dpi": 160,
})

BLUE = "#2F6F9F"
GREEN = "#4B8F5A"
RED = "#C84A3A"
GOLD = "#D9A441"
PURPLE = "#7B4EA3"
GRAY = "#4B5563"
LIGHT = "#F8FAFC"

def save(fig, name):
    path = OUT / name
    fig.savefig(path, bbox_inches="tight", dpi=220)
    plt.close(fig)
    print(path.relative_to(ROOT))

def add_box(ax, xy, wh, text, color, fontsize=12, face="white"):
    x, y = xy
    w, h = wh
    patch = FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.018,rounding_size=0.025",
        linewidth=1.6,
        edgecolor=color,
        facecolor=face,
    )
    ax.add_patch(patch)
    ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", fontsize=fontsize, color="#111827")

def add_panel(ax, xy, wh, color, face="white"):
    x, y = xy
    w, h = wh
    patch = FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.018,rounding_size=0.025",
        linewidth=1.6,
        edgecolor=color,
        facecolor=face,
    )
    ax.add_patch(patch)

def add_arrow(ax, start, end, color=GRAY):
    ax.add_patch(FancyArrowPatch(start, end, arrowstyle="-|>", mutation_scale=18, lw=1.8, color=color))


# 1) Clean SNPs, genotype coding, and design matrix schematic
fig, ax = plt.subplots(figsize=(12.6, 5.2))
ax.set_axis_off()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

add_panel(ax, (0.035, 0.56), (0.28, 0.34), BLUE, face=LIGHT)
ax.text(0.175, 0.855, "DNA sequence", ha="center", va="center", fontsize=13, fontweight="bold", color="#111827")
seq1 = list("A C G T A C T G A C".split())
seq2 = list("A C G T G C T G A T".split())
xs = np.linspace(0.075, 0.275, len(seq1))
for row, seq in enumerate([seq1, seq2]):
    y = 0.765 - row * 0.095
    ax.text(0.058, y, f"bird {row + 1}", ha="right", va="center", fontsize=9.5, color=GRAY)
    for x, base in zip(xs, seq):
        ax.text(x, y, base, ha="center", va="center", fontsize=13, fontweight="bold", color="#111827")
for idx in [4, 9]:
    ax.add_patch(Rectangle((xs[idx] - 0.012, 0.625), 0.024, 0.19, facecolor=RED, alpha=0.18, edgecolor=RED, lw=1.0))
ax.text(0.175, 0.605, "SNP: one position with different bases", ha="center", va="center", fontsize=10.5, color=RED)

add_panel(ax, (0.365, 0.56), (0.23, 0.34), GREEN, face="white")
ax.text(0.480, 0.855, "Genotype coding", ha="center", va="center", fontsize=13, fontweight="bold", color="#111827")
coding = [("AA", "0"), ("AG", "1"), ("GG", "2")]
for i, (geno, value) in enumerate(coding):
    y = 0.785 - i * 0.075
    ax.add_patch(Rectangle((0.415, y - 0.025), 0.055, 0.048, facecolor=LIGHT, edgecolor="#CBD5E1", lw=1.0))
    ax.add_patch(Rectangle((0.505, y - 0.025), 0.045, 0.048, facecolor=LIGHT, edgecolor="#CBD5E1", lw=1.0))
    ax.text(0.442, y, geno, ha="center", va="center", fontsize=11, color="#111827")
    ax.text(0.528, y, value, ha="center", va="center", fontsize=11, fontweight="bold", color="#111827")
ax.text(0.480, 0.592, "0, 1, 2 = allele count", ha="center", va="center", fontsize=10.5, color=GRAY)

add_panel(ax, (0.075, 0.09), (0.52, 0.34), GREEN, face="white")
ax.text(0.335, 0.390, "Design matrix $X$", ha="center", va="center", fontsize=13, fontweight="bold", color="#111827")
cols = ["SNP 1", "SNP 2", "SNP 3", "SNP 4", "SNP 5"]
rows = ["bird 1", "bird 2", "bird 3", "bird 4"]
mat = np.array([[0, 1, 2, 0, 1], [1, 1, 0, 2, 0], [2, 0, 1, 1, 2], [0, 2, 2, 1, 1]])
x0, y0, cw, ch = 0.178, 0.155, 0.067, 0.045
for c, label in enumerate(cols):
    ax.add_patch(Rectangle((x0 + c * cw, y0 + 4 * ch), cw, ch, facecolor="#EAF3F8", edgecolor="#CBD5E1", lw=0.9))
    ax.text(x0 + c * cw + cw / 2, y0 + 4.5 * ch, label, ha="center", va="center", fontsize=8.7, color=GRAY)
for r, label in enumerate(rows):
    y = y0 + (3 - r) * ch
    ax.text(0.155, y + ch / 2, label, ha="right", va="center", fontsize=9.0, color=GRAY)
    for c in range(mat.shape[1]):
        ax.add_patch(Rectangle((x0 + c * cw, y), cw, ch, facecolor="white", edgecolor="#CBD5E1", lw=0.9))
        ax.text(x0 + c * cw + cw / 2, y + ch / 2, str(mat[r, c]), ha="center", va="center", fontsize=10.5, color="#111827")
ax.text(0.335, 0.118, "many rows, many more SNP columns", ha="center", va="center", fontsize=10.5, color=GRAY)

add_panel(ax, (0.690, 0.12), (0.22, 0.31), GOLD, face="white")
ax.text(0.800, 0.385, "Phenotype $y$", ha="center", va="center", fontsize=13, fontweight="bold", color="#111827")
phen = ["0.42", "-0.18", "0.65", "-0.27"]
for r, (label, value) in enumerate(zip(rows, phen)):
    y = 0.330 - r * 0.050
    ax.text(0.735, y, label, ha="right", va="center", fontsize=9.0, color=GRAY)
    ax.add_patch(Rectangle((0.755, y - 0.022), 0.085, 0.043, facecolor=LIGHT, edgecolor="#CBD5E1", lw=0.9))
    ax.text(0.798, y, value, ha="center", va="center", fontsize=10.5, color="#111827")
ax.text(0.800, 0.128, "one measured trait", ha="center", va="center", fontsize=9.4, color=GRAY)

add_arrow(ax, (0.315, 0.73), (0.365, 0.73))
add_arrow(ax, (0.480, 0.56), (0.360, 0.43))
add_arrow(ax, (0.595, 0.265), (0.690, 0.265))
save(fig, "snp_to_prediction_schematic.png")


# 2) Training set optimization flow
fig, ax = plt.subplots(figsize=(10.6, 4.4))
ax.set_axis_off()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
add_box(ax, (0.04, 0.58), (0.20, 0.25), "Source islands\nphenotypes + SNPs", BLUE)
add_box(ax, (0.41, 0.58), (0.20, 0.25), "Select or weight\ntraining birds", GREEN)
add_box(ax, (0.77, 0.58), (0.19, 0.25), "Target island\nSNPs only", RED)
add_arrow(ax, (0.24, 0.705), (0.41, 0.705))
add_arrow(ax, (0.61, 0.705), (0.77, 0.705))
ax.text(0.15, 0.30, "$S$: candidate training set", ha="center", fontsize=14)
ax.text(0.51, 0.30, "$\\max$ predictive accuracy\nunder target-aware choice", ha="center", fontsize=14)
ax.text(0.865, 0.30, "$T$: birds we want\nto predict well", ha="center", fontsize=14)
save(fig, "training_set_optimization_flow.png")


# 3) Nested CV schematic
fig, ax = plt.subplots(figsize=(11.0, 4.9))
ax.set_axis_off()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
add_box(ax, (0.04, 0.62), (0.20, 0.23), "Outer loop\nhold out one island", BLUE, face=LIGHT)
add_box(ax, (0.36, 0.62), (0.25, 0.23), "Inner loop\nchoose method + hyperparameters", GREEN, face="white")
add_box(ax, (0.75, 0.62), (0.20, 0.23), "Final test\nheld-out target island", RED, face=LIGHT)
add_arrow(ax, (0.24, 0.735), (0.36, 0.735))
add_arrow(ax, (0.61, 0.735), (0.75, 0.735))
for i, x in enumerate(np.linspace(0.39, 0.58, 5)):
    color = GREEN if i < 4 else GOLD
    ax.add_patch(Rectangle((x, 0.41), 0.025, 0.09, facecolor=color, alpha=0.8, edgecolor="white"))
ax.text(0.485, 0.34, "repeat leave-one-island-out\ninside the source islands", ha="center", fontsize=12, color=GRAY)
ax.text(0.14, 0.36, "No target phenotype\nused for model choice", ha="center", fontsize=12, color=GRAY)
ax.text(0.85, 0.36, "One honest estimate\nof transfer performance", ha="center", fontsize=12, color=GRAY)
ax.text(0.50, 0.13, "Nested CV prevents tuning the training-set rule directly on the island we later report as test performance.", ha="center", fontsize=14, color="#111827")
save(fig, "nested_cv_schematic.png")


# 4) Top-k training-size mean curves
topk_specs = [
    (ROOT / "outputs" / "bpcrr_inla_2.0" / "bpcrr_inla_rank_select_results.csv", "bpcrr_topk_avggrm", "BPCRR avgGRM"),
    (ROOT / "outputs" / "avggrm_rank_weight_all" / "body_mass" / "avggrm_rank_weight_results.csv", "avggrm_topk", "Ridge avgGRM"),
    (ROOT / "outputs" / "pca_source_rank_weight_test4" / "pca_source_rank_weight_results.csv", "pca_source_topk", "Ridge PC distance"),
    (ROOT / "outputs" / "ridge_pev" / "pevmean_ga_results.csv", "pevmean_ga", "Ridge PEV-GA"),
]
frames = []
for path, method, label in topk_specs:
    df = pd.read_csv(path)
    df = df[df["method"] == method].copy()
    if "trait" in df.columns:
        df = df[df["trait"] == "body_mass"].copy()
    df = df[["target_island", "target_island_name", "n_individuals", "corr_eval"]].copy()
    df["method_label"] = label
    frames.append(df)
topk = pd.concat(frames, ignore_index=True)
common_sizes = sorted(set.intersection(*(set(f["n_individuals"].dropna().astype(int)) for f in frames)))
common_sizes = [s for s in common_sizes if 500 <= s <= 4500]
topk = topk[topk["n_individuals"].isin(common_sizes)].copy()
method_order = ["BPCRR avgGRM", "Ridge avgGRM", "Ridge PC distance", "Ridge PEV-GA"]
palette = {
    "BPCRR avgGRM": RED,
    "Ridge avgGRM": GREEN,
    "Ridge PC distance": BLUE,
    "Ridge PEV-GA": GOLD,
}
markers = {"BPCRR avgGRM": "o", "Ridge avgGRM": "s", "Ridge PC distance": "^", "Ridge PEV-GA": "D"}
topk_summary = (
    topk.groupby(["method_label", "n_individuals"], as_index=False)
    .agg(mean_corr=("corr_eval", "mean"), n=("corr_eval", "size"))
)

fig, ax = plt.subplots(figsize=(11.2, 5.2))
for label in method_order:
    sub = topk_summary[topk_summary["method_label"] == label].sort_values("n_individuals")
    ax.plot(
        sub["n_individuals"],
        sub["mean_corr"],
        label=label,
        color=palette[label],
        marker=markers[label],
        lw=2.5,
        ms=6.0,
    )
ax.axhline(0, color="#6B7280", lw=1)
ax.set_title("Top-k performance changes with training set size")
ax.set_xlabel("Training set size k")
ax.set_ylabel("Mean Pearson r across target islands")
ax.set_xticks(common_sizes)
ax.set_xticklabels([str(s) for s in common_sizes], rotation=0)
ax.legend(title=None, ncol=2, loc="upper left", frameon=True)
ax.set_ylim(-0.02, 0.34)
ax.grid(axis="x", visible=False)
save(fig, "topk_training_size_lines.png")


# 5) Data Shapley removal curve
shap = pd.read_csv(ROOT / "outputs" / "tmc_shapley_sweep_all_traits" / "body_mass" / "remove_curve_sweep_summary.csv")
shap_plot = shap[(shap["n_permutations"] == 150) & (shap["n_cal_samples"] == 80)].copy()
method_label = {"shapley_mean": "remove low-Shapley islands", "random_individual": "random removal"}
shap_plot["Method"] = shap_plot["method"].map(method_label)

fig, ax = plt.subplots(figsize=(8.8, 4.9))
colors = {"remove low-Shapley islands": RED, "random removal": GRAY}
for method, group in shap_plot.groupby("Method"):
    group = group.sort_values("n_removed")
    se = group["corr_std"] / np.sqrt(group["n_rows"].clip(lower=1))
    ax.plot(group["n_removed"], group["corr_mean"], label=method, color=colors[method], lw=2.4)
    ax.fill_between(group["n_removed"], group["corr_mean"] - se, group["corr_mean"] + se, color=colors[method], alpha=0.16, linewidth=0)
best = shap_plot[shap_plot["method"] == "shapley_mean"].loc[lambda d: d["corr_mean"].idxmax()]
base = shap_plot[(shap_plot["method"] == "shapley_mean") & (shap_plot["n_removed"] == 0)].iloc[0]
ax.scatter([best["n_removed"]], [best["corr_mean"]], s=130, color=RED, edgecolor="white", zorder=5)
ax.annotate(f"best: remove {int(best['n_removed'])}\n+{best['corr_mean']-base['corr_mean']:.3f} r", xy=(best["n_removed"], best["corr_mean"]), xytext=(best["n_removed"]+1.4, best["corr_mean"]+0.01), arrowprops=dict(arrowstyle="->", color=RED), fontsize=10)
ax.set_title("Data Shapley: small gains when removing harmful source islands")
ax.set_xlabel("Number of source islands removed")
ax.set_ylabel("Pearson correlation, body mass")
ax.legend(frameon=True)
ax.set_ylim(0.11, 0.19)
save(fig, "shapley_remove_curve_bodymass.png")


# 6) Nested-CV/full-source comparison with boxplots
report = prepare_report_data(ROOT, topk_value=1500)
full = build_full_baseline_comparison(report)
full_label_map = {
    "Ridge (avgGRM) | full_source_unweighted": "Ridge\nfull-source",
    "BPCRR | full_source_unweighted": "BPCRR\nfull-source",
    "Ridge (density-ratio nested CV)": "Ridge\ndensity-ratio",
}
full_plot = full[full["model_label"].isin(full_label_map)].copy()
full_plot["Model"] = full_plot["model_label"].map(full_label_map)
order = list(full_label_map.values())
fig, ax = plt.subplots(figsize=(9.8, 5.3))
sns.boxplot(data=full_plot, x="Model", y="corr", order=order, color="#DCEAF7", fliersize=0, showmeans=True, meanprops={"marker": "D", "markerfacecolor": RED, "markeredgecolor": "white", "markersize": 6}, ax=ax)
sns.stripplot(data=full_plot, x="Model", y="corr", order=order, color="#111827", alpha=0.42, size=4.2, jitter=0.18, ax=ax)
ax.axhline(0, color="#6B7280", lw=1)
ax.set_title("Nested density-ratio weighting is close to full-source baselines")
ax.set_ylabel("Pearson r on held-out island")
ax.set_xlabel("")
ax.set_ylim(-0.08, 0.45)
ax.grid(axis="x", visible=False)
save(fig, "nested_cv_outer_fold_comparison.png")


figures\presentation\snp_to_prediction_schematic.png
figures\presentation\training_set_optimization_flow.png


figures\presentation\nested_cv_schematic.png


figures\presentation\topk_training_size_lines.png


figures\presentation\shapley_remove_curve_bodymass.png


figures\presentation\nested_cv_outer_fold_comparison.png
